# 별첨 G: 의료비 예측 베이스라인 노트북

**모델**: M2 - 5년 누적 의료비 예측 (LightGBM Tweedie Regression)

**목적**: JMDC 데이터로 가입자 개인의 향후 5년 누적 의료비를 예측하는 베이스라인 모델을 학습하고, Quantile Regression으로 불확실성을 정량화합니다.

**사용 데이터**: JMDC HIS DB (가상 샘플 100명, 5년)

**알고리즘**:
- 베이스라인: LightGBM with Tweedie objective
- 불확실성: Quantile Regression (10%, 50%, 90%)

**평가 지표**: RMSE, MAE, R², Calibration by decile

In [ ]:
# 의존성
# pip install lightgbm pandas numpy scikit-learn shap matplotlib

import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import matplotlib.pyplot as plt
import shap

np.random.seed(42)

## 1. 데이터 로드

실제 운영 시에는 Supabase의 `ml.feature_matrix`와 `ml.outcome_labels`에서 데이터를 가져옵니다.

In [ ]:
# Supabase 연결 예시 (운영 환경)
# from supabase import create_client
# supabase = create_client(URL, KEY)
# data = supabase.table('feature_matrix').select('*').eq('cohort_id', COHORT_ID).execute()

# 데모용: 가상 데이터 생성
n_samples = 5000
df = pd.DataFrame({
    'age_at_index': np.random.normal(55, 8, n_samples).astype(int).clip(40, 75),
    'sex_code': np.random.choice([1, 2], n_samples),
    'bmi': np.random.normal(24, 3.5, n_samples).clip(15, 40),
    'sbp_mmhg': np.random.normal(125, 15, n_samples).clip(90, 200),
    'ldl_mgdl': np.random.normal(120, 30, n_samples).clip(50, 250),
    'hba1c_ngsp': np.random.normal(5.7, 0.8, n_samples).clip(4.5, 12),
    'charlson_total': np.random.poisson(1.2, n_samples).clip(0, 10),
    'has_diabetes': np.random.binomial(1, 0.15, n_samples),
    'has_htn': np.random.binomial(1, 0.30, n_samples),
    'polypharmacy_flag': np.random.binomial(1, 0.20, n_samples),
    'n_outpatient_visits_prev': np.random.negative_binomial(5, 0.5, n_samples).clip(0, 50),
    'had_hospitalization_prev': np.random.binomial(1, 0.10, n_samples),
})

# 가상 의료비 생성 (5년 누적, 엔)
# 의료비는 right-skewed → Tweedie distribution 모방
base_cost = (
    df['age_at_index'] * 5000
    + df['charlson_total'] * 150000
    + df['has_diabetes'] * 200000
    + df['has_htn'] * 80000
    + df['polypharmacy_flag'] * 100000
    + df['had_hospitalization_prev'] * 500000
    + df['n_outpatient_visits_prev'] * 10000
)
noise = np.random.gamma(shape=2, scale=base_cost / 4)
df['total_cost_5y_yen'] = noise.clip(0, 30_000_000).astype(int)

print(f'Samples: {len(df):,}')
print(f'Cost stats:\n{df["total_cost_5y_yen"].describe()}')

## 2. 데이터 분할 (Train / Val / Test)

In [ ]:
feature_cols = [c for c in df.columns if c != 'total_cost_5y_yen']
X = df[feature_cols]
y = df['total_cost_5y_yen']

X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.2, random_state=42)

print(f'Train: {len(X_train):,}, Val: {len(X_val):,}, Test: {len(X_test):,}')
print(f'Train mean cost: {y_train.mean():,.0f} JPY')
print(f'Train median cost: {y_train.median():,.0f} JPY')

## 3. LightGBM Tweedie Regression 학습

Tweedie 분포는 의료비처럼 0이 많고 right-skewed인 데이터에 적합합니다.

In [ ]:
params = {
    'objective': 'tweedie',
    'tweedie_variance_power': 1.5,
    'metric': 'rmse',
    'learning_rate': 0.05,
    'num_leaves': 63,
    'min_data_in_leaf': 50,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'lambda_l2': 0.1,
    'verbose': -1,
}

train_set = lgb.Dataset(X_train, y_train)
val_set = lgb.Dataset(X_val, y_val, reference=train_set)

model = lgb.train(
    params,
    train_set,
    num_boost_round=2000,
    valid_sets=[val_set],
    callbacks=[
        lgb.early_stopping(100),
        lgb.log_evaluation(100),
    ],
)

## 4. 평가

In [ ]:
y_pred = model.predict(X_test, num_iteration=model.best_iteration)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
relative_rmse = rmse / y_test.mean()

print(f'\n=== Test Set Performance ===')
print(f'RMSE: {rmse:,.0f} JPY')
print(f'MAE: {mae:,.0f} JPY')
print(f'R²: {r2:.3f}')
print(f'Relative RMSE: {relative_rmse:.3f}')

# 임계값 검증
assert relative_rmse < 1.0, '베이스라인은 평균 대비 1배 RMSE 이하 목표'
print('\n[V] 베이스라인 성능 통과')

## 5. Bias by Decile (Calibration)

In [ ]:
decile_df = pd.DataFrame({'y_true': y_test, 'y_pred': y_pred})
decile_df['decile'] = pd.qcut(decile_df['y_pred'], 10, labels=False, duplicates='drop')

calibration = decile_df.groupby('decile').agg(
    mean_pred=('y_pred', 'mean'),
    mean_actual=('y_true', 'mean'),
    count=('y_true', 'count'),
).reset_index()
calibration['bias_pct'] = (calibration['mean_pred'] - calibration['mean_actual']) / calibration['mean_actual'] * 100

print('=== Bias by Predicted Decile ===')
print(calibration.to_string(index=False))

# 시각화
fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(calibration['mean_pred'], calibration['mean_actual'], 'o-', label='Decile mean')
ax.plot([0, calibration[['mean_pred', 'mean_actual']].max().max()],
        [0, calibration[['mean_pred', 'mean_actual']].max().max()], 'k--', label='Perfect calibration')
ax.set_xlabel('Mean Predicted Cost (JPY)')
ax.set_ylabel('Mean Actual Cost (JPY)')
ax.set_title('Calibration by Decile')
ax.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Quantile Regression for Uncertainty

각 가입자에 대해 10%, 50%, 90% 분위의 의료비 추정치를 산출합니다.

In [ ]:
quantile_models = {}
for q in [0.1, 0.5, 0.9]:
    print(f'\n[Training quantile {q}]')
    q_params = {
        'objective': 'quantile',
        'alpha': q,
        'metric': 'quantile',
        'learning_rate': 0.05,
        'num_leaves': 63,
        'min_data_in_leaf': 50,
        'verbose': -1,
    }
    qm = lgb.train(
        q_params,
        lgb.Dataset(X_train, y_train),
        num_boost_round=1000,
        valid_sets=[lgb.Dataset(X_val, y_val)],
        callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)],
    )
    quantile_models[q] = qm

# 예측
predictions = {q: m.predict(X_test, num_iteration=m.best_iteration) for q, m in quantile_models.items()}

result = pd.DataFrame({
    'actual': y_test.values,
    'p10': predictions[0.1],
    'p50_median': predictions[0.5],
    'p90': predictions[0.9],
})

# 90% 신뢰구간 커버리지 확인
in_ci = ((result['actual'] >= result['p10']) & (result['actual'] <= result['p90'])).mean()
print(f'\n90% prediction interval coverage: {in_ci:.1%} (목표 80%)')
print(f'\nSample predictions:')
print(result.head(10).to_string(index=False))

## 7. SHAP 분석 (피처 중요도)

In [ ]:
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test.iloc[:500])  # 샘플링

# 글로벌 중요도
shap.summary_plot(shap_values, X_test.iloc[:500], plot_type='bar', show=False)
plt.tight_layout()
plt.show()

# 개별 환자 설명
shap.force_plot(
    explainer.expected_value,
    shap_values[0],
    X_test.iloc[0],
    matplotlib=True,
    show=True
)

## 8. 보험 비즈니스 적용 예시

신청자 A의 5년 의료비 예측 → 진단보험금 보장 한도 결정

In [ ]:
applicant = pd.DataFrame({
    'age_at_index': [52],
    'sex_code': [1],
    'bmi': [26.4],
    'sbp_mmhg': [142],
    'ldl_mgdl': [148],
    'hba1c_ngsp': [6.4],
    'charlson_total': [2],
    'has_diabetes': [0],
    'has_htn': [1],
    'polypharmacy_flag': [0],
    'n_outpatient_visits_prev': [8],
    'had_hospitalization_prev': [0],
})

median_pred = quantile_models[0.5].predict(applicant)[0]
p10_pred = quantile_models[0.1].predict(applicant)[0]
p90_pred = quantile_models[0.9].predict(applicant)[0]

print(f'=== 신청자 A의 5년 의료비 예측 ===')
print(f'  중앙값 추정: {median_pred:,.0f} JPY')
print(f'  90% 신뢰구간: [{p10_pred:,.0f}, {p90_pred:,.0f}] JPY')
print(f'\n=== 보험 적용 ===')
print(f'  권장 보장한도 (P90 × 1.1): {p90_pred * 1.1:,.0f} JPY')
print(f'  예상 손해율 = 예측 의료비 / 보장한도')
print(f'\n주의: 본 결과는 모델 예측이며, 실제 보험 인수 결정에는 추가 전문가 검토 필요')

## 9. 모델 저장 + MLflow 로깅

실제 운영 시에는 MLflow에 자동 등록되어 모델 레지스트리에서 관리됩니다.

In [ ]:
# import mlflow
# import mlflow.lightgbm
# 
# mlflow.set_experiment('jmdc_cost_prediction')
# with mlflow.start_run() as run:
#     mlflow.log_params(params)
#     mlflow.log_metrics({'rmse': rmse, 'mae': mae, 'r2': r2})
#     mlflow.lightgbm.log_model(model, 'cost_model_lgbm')
#     print(f'MLflow run: {run.info.run_id}')

# 로컬 저장
model.save_model('cost_model_v1.txt')
for q, m in quantile_models.items():
    m.save_model(f'cost_model_q{int(q*100)}.txt')

print('모델 저장 완료')

---

## 다음 단계

1. **실제 JMDC 데이터로 재학습**: 본 노트북의 가상 데이터를 Supabase의 `ml.feature_matrix`로 교체
2. **하이퍼파라미터 튜닝**: Optuna 또는 Hyperopt로 자동 탐색
3. **외부 검증**: HIRA K-OMOP 데이터로 한국 가입자 적용 시 성능 평가
4. **프로덕션 배포**: FastAPI endpoint로 서빙, 모니터링 자동화

본 베이스라인을 출발점으로 PRD 6.2절 (M2 의료비 예측) 전체 구현을 진행하세요.